In [20]:
# from pyspark.sql import SparkSession, functions as F

# spark = (
#     SparkSession.builder
#     .appName("MinIO-Test")
#     .master("local[*]")
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
#     .getOrCreate()
# )

# hconf = spark.sparkContext._jsc.hadoopConfiguration()
# hconf.set("fs.s3a.endpoint", "http://minio:9000")
# hconf.set("fs.s3a.access.key", "matrix")
# hconf.set("fs.s3a.secret.key", "matrix123")
# hconf.set("fs.s3a.path.style.access", "true")
# hconf.set("fs.s3a.connection.ssl.enabled", "false")
# hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/postgresql-42.7.1.jar")  # PostgreSQL JDBC driver
    .getOrCreate()
)

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")


In [2]:
bronze_path = "s3a://bronze/"
silver_path = "s3a://silver/"
gold_path = "s3a://gold/"

In [3]:
df_card = spark.read.csv("card.csv", header=True, inferSchema=True)

In [4]:
df_customer= spark.read.csv("customer.csv", header=True, inferSchema=True)

In [5]:
df_trs= spark.read.csv("trs.csv", header=True, inferSchema=True)

In [6]:
df_card.show(5)

+-------+-----------+----------------+---------+-------+
|card_id|customer_id|     card_number|card_type| status|
+-------+-----------+----------------+---------+-------+
|   K001|       C001|4111111111111111|    DEBIT| ACTIVE|
|   K002|       C002|5222222222222222|   CREDIT| ACTIVE|
|   K003|       C003|6333333333333333|    DEBIT|BLOCKED|
|   K004|       C004|7444444444444444|   CREDIT| ACTIVE|
|   K005|       C005|8555555555555555|    DEBIT| ACTIVE|
+-------+-----------+----------------+---------+-------+



In [7]:
df_card.write.mode("overwrite").option("header", "true").csv(f"{bronze_path}/card.csv")

In [8]:
df_customer.write.mode("overwrite").option("header", "true").csv(f"{bronze_path}/customer.csv")

In [9]:
df_trs.write.mode("overwrite").option("header", "true").csv(f"{bronze_path}/trs.csv")

In [10]:
#################bronze bitdi

In [11]:
df_card.show(5)

+-------+-----------+----------------+---------+-------+
|card_id|customer_id|     card_number|card_type| status|
+-------+-----------+----------------+---------+-------+
|   K001|       C001|4111111111111111|    DEBIT| ACTIVE|
|   K002|       C002|5222222222222222|   CREDIT| ACTIVE|
|   K003|       C003|6333333333333333|    DEBIT|BLOCKED|
|   K004|       C004|7444444444444444|   CREDIT| ACTIVE|
|   K005|       C005|8555555555555555|    DEBIT| ACTIVE|
+-------+-----------+----------------+---------+-------+



In [20]:
jdbc_url = "jdbc:postgresql://postgres:5432/mydb"
jdbc_properties = {
    "user": "nurgun",
    "password": "admin123",  
    "driver": "org.postgresql.Driver"
}

df_card2 = spark.read.jdbc(url=jdbc_url, table="card", properties=jdbc_properties)

In [21]:
df_trs2= spark.read.jdbc(url=jdbc_url, table="transaction", properties=jdbc_properties)

In [22]:
df_customer2= spark.read.jdbc(url=jdbc_url, table="customer", properties=jdbc_properties)

In [23]:
df_card2.show(5)

+-------+-----------+----------------+---------+-----------+------------------+
|card_id|customer_id|     card_number|card_type|card_status|card_number_masked|
+-------+-----------+----------------+---------+-----------+------------------+
|   K001|       C001|4111111111111111|    DEBIT|     ACTIVE|  4111********1111|
|   K002|       C002|5222222222222222|   CREDIT|     ACTIVE|  5222********2222|
|   K003|       C003|6333333333333333|    DEBIT|    BLOCKED|  6333********3333|
|   K004|       C004|7444444444444444|   CREDIT|     ACTIVE|  7444********4444|
|   K005|       C005|8555555555555555|    DEBIT|     ACTIVE|  8555********5555|
+-------+-----------+----------------+---------+-----------+------------------+



In [24]:
df_card2.write.mode("overwrite").option("header", "true").csv(f"{silver_path}/card_cleaned.csv")
df_customer2.write.mode("overwrite").option("header", "true").csv(f"{silver_path}/customer_cleaned.csv")
df_trs2.write.mode("overwrite").option("header", "true").csv(f"{silver_path}/trs_cleaned.csv")

In [27]:
df_customer3 = spark.read.csv(f"{silver_path}/customer_cleaned.csv", header=True, inferSchema=True)
df_card3 = spark.read.csv(f"{silver_path}/card_cleaned.csv", header=True, inferSchema=True)
df_trs3 = spark.read.csv(f"{silver_path}/trs_cleaned.csv", header=True, inferSchema=True)

In [28]:
df_customer_non_vip = df_customer3.filter(F.col("flag_is_vip") == False)

In [33]:
df_gold = df_customer_non_vip.alias("c") \
    .join(df_card3.alias("card"), F.col("c.customer_id") == F.col("card.customer_id"), "left") \
    .join(df_trs3.alias("t"), F.col("card.card_id") == F.col("t.card_id"), "left")



In [38]:
df_joined = df_customer_non_vip \
    .join(df_card3, "customer_id", "left") \
    .join(df_trs3, "card_id", "left")

In [39]:
df_datamart = df_joined.groupBy("customer_id") \
    .agg(
        F.current_date().alias("etl_date"),
        F.countDistinct("card_id").alias("cards_count"),
        F.sum(F.when(F.col("card_status") == "ACTIVE", 1).otherwise(0)).alias("active_cards_count"),
        F.sum("amount").alias("total_amount_all_time"),
        F.count("transaction_id").alias("transaction_count_all_time"),
        F.min("transaction_date").alias("first_transaction_date"),
        F.max("transaction_date").alias("last_transaction_date")
    ) \
    .withColumn("days_since_last_txn", 
                F.datediff(F.current_date(), F.col("last_transaction_date")))

In [40]:
df_datamart.show()

+-----------+----------+-----------+------------------+---------------------+--------------------------+----------------------+---------------------+-------------------+
|customer_id|  etl_date|cards_count|active_cards_count|total_amount_all_time|transaction_count_all_time|first_transaction_date|last_transaction_date|days_since_last_txn|
+-----------+----------+-----------+------------------+---------------------+--------------------------+----------------------+---------------------+-------------------+
|       C003|2026-02-07|          1|                 0|                325.0|                         7|            2025-01-05|           2025-01-18|                385|
|       C004|2026-02-07|          1|                12|               4170.0|                        12|            2025-01-02|           2025-01-19|                384|
|       C005|2026-02-07|          1|                10|               2745.0|                        10|            2025-01-02|           2025-01-18| 

In [41]:
df_datamart.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- etl_date: date (nullable = false)
 |-- cards_count: long (nullable = false)
 |-- active_cards_count: long (nullable = true)
 |-- total_amount_all_time: double (nullable = true)
 |-- transaction_count_all_time: long (nullable = false)
 |-- first_transaction_date: date (nullable = true)
 |-- last_transaction_date: date (nullable = true)
 |-- days_since_last_txn: integer (nullable = true)



In [42]:
df_datamart.write.mode("overwrite").option("header", "true").csv(f"{gold_path}/customer_datamart.csv")

In [48]:
df_datamart.show()

+-----------+----------+-----------+------------------+---------------------+--------------------------+----------------------+---------------------+-------------------+
|customer_id|  etl_date|cards_count|active_cards_count|total_amount_all_time|transaction_count_all_time|first_transaction_date|last_transaction_date|days_since_last_txn|
+-----------+----------+-----------+------------------+---------------------+--------------------------+----------------------+---------------------+-------------------+
|       C003|2026-02-07|          1|                 0|                325.0|                         7|            2025-01-05|           2025-01-18|                385|
|       C004|2026-02-07|          1|                12|               4170.0|                        12|            2025-01-02|           2025-01-19|                384|
|       C005|2026-02-07|          1|                10|               2745.0|                        10|            2025-01-02|           2025-01-18| 